In [144]:
# SERVE SOLO AL JUPITER NOTEBOOK
#----------------------------------

import getpass
# 0. Inserimento sicuro del Token JWT (evita di salvarlo in chiaro nel notebook)
if "JWT" not in os.environ:
    os.environ["JWT"] = getpass.getpass("Inserisci il tuo Token JWT: ")

# Assicuriamo che sia impostata anche OPENAI_API_KEY (usata di default da lm_eval)
os.environ["OPENAI_API_KEY"] = os.environ["JWT"]

#-----------------------------------

In [81]:
import run_benchmarks as bench
import json
import os

In [82]:
def add_setting(task, settings:dict, edit=False):
    with open("./task-settings.json", "r") as f:
        settings_dict = json.load(f)
        if not edit and task in settings_dict.keys():
            raise Exception("add_setting not in edit mode! ")
        else:
            settings_dict[task] = settings
    with open("./task-settings.json", "w") as f:
        json.dump(settings_dict, f, indent=4, ensure_ascii=False)


In [83]:
settings_file = "./task-settings.json"
models_file = "./models.json"
with open(settings_file, "r") as f:
    task_settings = json.load(f)
with open(models_file, "r") as f:
    models = json.load(f)["models"]

In [124]:
# AIME25

task_s = task_settings["chat-tests"]

task_s["gen_kwargs"]["do_sample"] = False
task_s["gen_kwargs"]["max_tokens"] = 32768 * 4
task_s["num_fewshot"] = 0
task_s["tasks"] = ["aime25"]
model = "Qwen/Qwen3.8-27B-FP8"
task_s["model_args"]["model"] = "Qwen/Qwen3.8-27B-FP8"


In [186]:
task_s["limit"] = 5
task_s["log_samples"] = True
task_s["max_gen_toks"] = task_s["gen_kwargs"]["max_tokens"]

# alziamo la temperatura per vedere se così finisce prima il reasoning
task_s["gen_kwargs"]["temperature"] = 0.6
task_s["gen_kwargs"]["top_p"] = 0.95


outp = bench.run_benchmark(task_s)

generation_kwargs: {'max_tokens': 131072, 'temperature': 0.6, 'until': [], 'do_sample': False, 'extra_body': {'include_reasoning': True}, 'top_p': 0.95} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
100%|██████████| 5/5 [00:00<00:00, 163.84it/s]
Cannot determine EOS string to pass to stop sequence. Manually set by passing `eos_string` to model_args.
API returned null content. Content filled with `LMEVAL_MODEL_NONE_ANSWER_PLACEHOLDER = `. Check reasoning_content field or generation limits.
Requesting API: 100%|██████████| 5/5 [03:58<00:00, 47.64s/it]


In [219]:
outp["samples"]["aime25"][0]

{'doc_id': 0,
 'doc': {'problem': 'Find the sum of all integer bases $b>9$ for which $17_b$ is a divisor of $97_b.$',
  'answer': '70',
  'id': '0'},
 'target': '70',
 'arguments': [(('[{"role": "user", "content": "Question: Find the sum of all integer bases $b>9$ for which $17_b$ is a divisor of $97_b.$\\nAnswer:"}]',),
   {'until': [],
    'do_sample': False,
    'temperature': 0.6,
    'max_gen_toks': 32768,
    'max_tokens': 131072,
    'extra_body': {'include_reasoning': True},
    'top_p': 0.95})],
 'resps': [['\n\nWe have\n\n\\[\n17_b = b+7\n\\]\n\nand\n\n\\[\n97_b = 9b+7.\n\\]\n\nWe need \\(b+7 \\mid 9b+7\\). Rewrite:\n\n\\[\n9b+7 = 9(b+7)-56.\n\\]\n\nSo \\(b+7\\) must divide \\(56\\). Since \\(b>9\\),\n\n\\[\nb+7>16.\n\\]\n\nThe divisors of \\(56\\) greater than \\(16\\) are \\(28\\) and \\(56\\). Thus\n\n\\[\nb+7=28 \\implies b=21,\n\\]\n\\[\nb+7=56 \\implies b=49.\n\\]\n\nThe sum of all such bases is\n\n\\[\n21+49=70.\n\\]\n\n\\[\n\\boxed{70}\n\\]']],
 'filtered_resps': ['\n

In [212]:
for j in range(len(outp["samples"]["aime25"])):
    print( outp["samples"]["aime25"][j]["resps"] )

[['\n\nWe have\n\n\\[\n17_b = b+7\n\\]\n\nand\n\n\\[\n97_b = 9b+7.\n\\]\n\nWe need \\(b+7 \\mid 9b+7\\). Rewrite:\n\n\\[\n9b+7 = 9(b+7)-56.\n\\]\n\nSo \\(b+7\\) must divide \\(56\\). Since \\(b>9\\),\n\n\\[\nb+7>16.\n\\]\n\nThe divisors of \\(56\\) greater than \\(16\\) are \\(28\\) and \\(56\\). Thus\n\n\\[\nb+7=28 \\implies b=21,\n\\]\n\\[\nb+7=56 \\implies b=49.\n\\]\n\nThe sum of all such bases is\n\n\\[\n21+49=70.\n\\]\n\n\\[\n\\boxed{70}\n\\]']]
[['']]
[['\n\nLet the numbers of players choosing chocolate, vanilla, and strawberry be \\(c,v,s\\), respectively. We need\n\n\\[\nc+v+s=9,\\qquad c>v>s\\ge 1.\n\\]\n\nThe possible triples \\((c,v,s)\\) are:\n\n- \\(s=1\\): \\((6,2,1)\\), \\((5,3,1)\\)\n- \\(s=2\\): \\((4,3,2)\\)\n\nFor each triple, the number of assignments to the 9 distinct players is given by a multinomial coefficient.\n\n\\[\n\\frac{9!}{6!2!1!}=252\n\\]\n\n\\[\n\\frac{9!}{5!3!1!}=504\n\\]\n\n\\[\n\\frac{9!}{4!3!2!}=1260\n\\]\n\nThus\n\n\\[\nN=252+504+1260=2016.\n\\]\n

In [216]:
outp["samples"]["aime25"][1]

{'doc_id': 1,
 'doc': {'problem': 'In $\\triangle ABC$ points $D$ and $E$ lie on $\\overline{AB}$ so that $AD < AE < AB$, while points $F$ and $G$ lie on $\\overline{AC}$ so that $AF < AG < AC$. Suppose $AD = 4$, $DE = 16$, $EB = 8$, $AF = 13$, $FG = 52$, and $GC = 26$. Let $M$ be the reflection of $D$ through $F$, and let $N$ be the reflection of $G$ through $E$. The area of quadrilateral $DEGF$ is $288$. Find the area of heptagon $AFNBCEM$.',
  'answer': '588',
  'id': '1'},
 'target': '588',
 'arguments': [(('[{"role": "user", "content": "Question: In $\\\\triangle ABC$ points $D$ and $E$ lie on $\\\\overline{AB}$ so that $AD < AE < AB$, while points $F$ and $G$ lie on $\\\\overline{AC}$ so that $AF < AG < AC$. Suppose $AD = 4$, $DE = 16$, $EB = 8$, $AF = 13$, $FG = 52$, and $GC = 26$. Let $M$ be the reflection of $D$ through $F$, and let $N$ be the reflection of $G$ through $E$. The area of quadrilateral $DEGF$ is $288$. Find the area of heptagon $AFNBCEM$.\\nAnswer:"}]',),
   {'un

In [137]:
outp["samples"]["aime25"][1]

{'doc_id': 1,
 'doc': {'problem': 'In $\\triangle ABC$ points $D$ and $E$ lie on $\\overline{AB}$ so that $AD < AE < AB$, while points $F$ and $G$ lie on $\\overline{AC}$ so that $AF < AG < AC$. Suppose $AD = 4$, $DE = 16$, $EB = 8$, $AF = 13$, $FG = 52$, and $GC = 26$. Let $M$ be the reflection of $D$ through $F$, and let $N$ be the reflection of $G$ through $E$. The area of quadrilateral $DEGF$ is $288$. Find the area of heptagon $AFNBCEM$.',
  'answer': '588',
  'id': '1'},
 'target': '588',
 'arguments': [(('[{"role": "user", "content": "Question: In $\\\\triangle ABC$ points $D$ and $E$ lie on $\\\\overline{AB}$ so that $AD < AE < AB$, while points $F$ and $G$ lie on $\\\\overline{AC}$ so that $AF < AG < AC$. Suppose $AD = 4$, $DE = 16$, $EB = 8$, $AF = 13$, $FG = 52$, and $GC = 26$. Let $M$ be the reflection of $D$ through $F$, and let $N$ be the reflection of $G$ through $E$. The area of quadrilateral $DEGF$ is $288$. Find the area of heptagon $AFNBCEM$.\\nAnswer:"}]',),
   {'un

In [128]:
outp["samples"]["aime25"][1]["resps"][0][0] == ''


True

In [142]:
print(outp["samples"]["aime25"][1]["arguments"][0][0][0])

[{"role": "user", "content": "Question: In $\\triangle ABC$ points $D$ and $E$ lie on $\\overline{AB}$ so that $AD < AE < AB$, while points $F$ and $G$ lie on $\\overline{AC}$ so that $AF < AG < AC$. Suppose $AD = 4$, $DE = 16$, $EB = 8$, $AF = 13$, $FG = 52$, and $GC = 26$. Let $M$ be the reflection of $D$ through $F$, and let $N$ be the reflection of $G$ through $E$. The area of quadrilateral $DEGF$ is $288$. Find the area of heptagon $AFNBCEM$.\nAnswer:"}]


In [189]:
import json
import requests

url = "https://orfeo-llm.areasciencepark.it/vllm/v1/chat/completions"

# Il prefisso r""" evita che $ o \ vengano interpretati dalla shell o da Python
aime_prompt = r"""Question: In $\triangle ABC$ points $D$ and $E$ lie on $\overline{AB}$ so that $AD < AE < AB$, while points $F$ and $G$ lie on $\overline{AC}$ so that $AF < AG < AC$. Suppose $DE = 4$, $EB = 16$, $FG = 8$, $GC = 13$, $EF = 52$, and $DG = 26$. Let $P$ be the reflection of $A$ through $DG$, and let $Q$ be the reflection of $A$ through $EF$. The area of quadrilateral $DEFG$ is $88$. Find the area of heptagon $ABCQP$.Answer:"""

API_KEY = os.environ["JWT"]

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}", 
}

payload = {
    "model": task_s["model_args"]["model"],
    "messages": [{"role": "user", "content": aime_prompt}],
    "max_tokens": task_s["gen_kwargs"]["max_tokens"],
    "temperature": 0.6,
    "top_p": 0.95
}

response = requests.post(
    url, headers=headers, json=payload
)

print(f"Status Code HTTP: {response.status_code}\n")

if response.status_code == 200:
    print(json.dumps(response.json(), indent=2))
else:
    print("Errore dal server:")
    print(response.text)

Status Code HTTP: 200

{
  "id": "chatcmpl-9ffd33985848a06f",
  "object": "chat.completion",
  "created": 1789743736,
  "model": "Qwen/Qwen3.8-27B-FP8",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": null,
        "refusal": null,
        "annotations": null,
        "audio": null,
        "function_call": null,
        "reasoning": "We need solve geometry problem. We need provide final answer. Let's analyze thoroughly.\n\nWe have triangle ABC. Points D,E on AB with AD < AE < AB. F,G on AC with AF < AG < AC. Given lengths:\nDE=4, EB=16, FG=8, GC=13, EF=52, DG=26.\nArea quadrilateral DEFG = 88.\nP reflection of A through DG, Q reflection of A through EF.\nFind area of heptagon ABCQP.\n\nWe need infer configuration. Points D,E on AB, F,G on AC. Quadrilateral DEFG? Order D-E-F-G? D and E on one side, F and G on other. Given DE along AB, FG along AC, EF and DG cross? Let's set coordinate.\n\nLet A at origin. Let AB be x-axis? Since

In [ ]:
resp = json.loads(response.text)


In [194]:
for k in resp.keys():
    print(k)

id
object
created
model
choices
service_tier
system_fingerprint
usage
prompt_logprobs
prompt_token_ids
prompt_text
kv_transfer_params
ec_transfer_params
metrics


In [195]:
resp["usage"]

{'prompt_tokens': 210,
 'total_tokens': 10210,
 'completion_tokens': 10000,
 'prompt_tokens_details': None}

In [192]:
resp["choices"][0]["message"]["reasoning"]

'We need solve geometry problem. We need provide final answer. Let\'s analyze thoroughly.\n\nWe have triangle ABC. Points D,E on AB with AD < AE < AB. F,G on AC with AF < AG < AC. Given lengths:\nDE=4, EB=16, FG=8, GC=13, EF=52, DG=26.\nArea quadrilateral DEFG = 88.\nP reflection of A through DG, Q reflection of A through EF.\nFind area of heptagon ABCQP.\n\nWe need infer configuration. Points D,E on AB, F,G on AC. Quadrilateral DEFG? Order D-E-F-G? D and E on one side, F and G on other. Given DE along AB, FG along AC, EF and DG cross? Let\'s set coordinate.\n\nLet A at origin. Let AB be x-axis? Since D,E on AB. Let AB along positive x-axis. Let AC be another ray from A making angle theta. Points: D at distance d=AD, E at distance e=AE, with d<e. DE = e-d =4. EB = AB - e =16. So AB = e+16. Let e = d+4, AB = d+20.\n\nOn AC: F at distance f=AF, G at distance g=AG, f<g. FG = g-f=8. GC = AC - g =13. AC = g+13 = f+21.\n\nGiven EF=52, DG=26. Quadrilateral DEFG area = 88. Need area of heptago

In [193]:
resp

{'id': 'chatcmpl-9ffd33985848a06f',
 'object': 'chat.completion',
 'created': 1789743736,
 'model': 'Qwen/Qwen3.8-27B-FP8',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': None,
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'reasoning': 'We need solve geometry problem. We need provide final answer. Let\'s analyze thoroughly.\n\nWe have triangle ABC. Points D,E on AB with AD < AE < AB. F,G on AC with AF < AG < AC. Given lengths:\nDE=4, EB=16, FG=8, GC=13, EF=52, DG=26.\nArea quadrilateral DEFG = 88.\nP reflection of A through DG, Q reflection of A through EF.\nFind area of heptagon ABCQP.\n\nWe need infer configuration. Points D,E on AB, F,G on AC. Quadrilateral DEFG? Order D-E-F-G? D and E on one side, F and G on other. Given DE along AB, FG along AC, EF and DG cross? Let\'s set coordinate.\n\nLet A at origin. Let AB be x-axis? Since D,E on AB. Let AB along positive x-axis. Let AC be another ray from A makin

In [196]:
task_s["gen_kwargs"]["max_tokens"]

131072